In [ ]:
import sys
import subprocess

def ensure_package(pkg, import_name=None):
    name = import_name or pkg
    try:
        __import__(name)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg])

ensure_package('datasets')
ensure_package('pandas')
ensure_package('numpy')
ensure_package('scikit-learn', 'sklearn')

In [ ]:
import random
import time
import numpy as np
import pandas as pd

from datasets import load_dataset
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
pd.set_option('display.max_colwidth', 200)
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 50)

In [ ]:
dataset = load_dataset('emotion')
print(dataset)
print('Splits:', list(dataset.keys()))
for split in dataset.keys():
    print(split, len(dataset[split]))

In [ ]:
label_feature = dataset['train'].features['label']
label_names = label_feature.names
id2label = {i: name for i, name in enumerate(label_names)}
label2id = {name: i for i, name in id2label.items()}
print('Labels:', id2label)

train_df = dataset['train'].to_pandas()
val_df = dataset['validation'].to_pandas()
test_df = dataset['test'].to_pandas()

for df in [train_df, val_df, test_df]:
    df['label_name'] = df['label'].map(id2label)
    df['text_clean'] = df['text'].astype(str).str.strip().str.lower()
    df['char_len'] = df['text'].astype(str).str.len()
    df['word_len'] = df['text'].astype(str).str.split().str.len()

print(train_df.head(10))

In [ ]:
print('Train label distribution:')
print(train_df['label_name'].value_counts().sort_index())
print('\nValidation label distribution:')
print(val_df['label_name'].value_counts().sort_index())
print('\nTest label distribution:')
print(test_df['label_name'].value_counts().sort_index())

print('\nText length stats (train):')
print(train_df[['char_len', 'word_len']].describe())

In [ ]:
X_train = train_df['text_clean'].tolist()
y_train = train_df['label'].tolist()
X_val = val_df['text_clean'].tolist()
y_val = val_df['label'].tolist()
X_test = test_df['text_clean'].tolist()
y_test = test_df['label'].tolist()

model = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.95, sublinear_tf=True)),
    ('clf', LogisticRegression(max_iter=1000, random_state=SEED, n_jobs=None))
])

start_train = time.time()
model.fit(X_train, y_train)
train_time = time.time() - start_train

start_val = time.time()
val_preds = model.predict(X_val)
val_probas = model.predict_proba(X_val)
val_infer_time = time.time() - start_val

start_test = time.time()
test_preds = model.predict(X_test)
test_probas = model.predict_proba(X_test)
test_infer_time = time.time() - start_test

val_acc = accuracy_score(y_val, val_preds)
val_f1 = f1_score(y_val, val_preds, average='macro')
test_acc = accuracy_score(y_test, test_preds)
test_f1 = f1_score(y_test, test_preds, average='macro')

print(f'Training time: {train_time:.3f}s')
print(f'Validation inference time: {val_infer_time:.3f}s')
print(f'Test inference time: {test_infer_time:.3f}s')
print(f'Validation Accuracy: {val_acc:.4f}')
print(f'Validation Macro F1: {val_f1:.4f}')
print(f'Test Accuracy: {test_acc:.4f}')
print(f'Test Macro F1: {test_f1:.4f}')

In [ ]:
print('Validation classification report:')
print(classification_report(y_val, val_preds, target_names=label_names, digits=4))

print('Test classification report:')
print(classification_report(y_test, test_preds, target_names=label_names, digits=4))

cm = confusion_matrix(y_test, test_preds)
cm_df = pd.DataFrame(cm, index=[f'true_{x}' for x in label_names], columns=[f'pred_{x}' for x in label_names])
print('Test confusion matrix:')
print(cm_df)

In [ ]:
results_df = pd.DataFrame([
    {
        'model': 'TF-IDF + LogisticRegression',
        'val_accuracy': val_acc,
        'val_macro_f1': val_f1,
        'test_accuracy': test_acc,
        'test_macro_f1': test_f1,
        'train_time_sec': train_time,
        'val_inference_sec': val_infer_time,
        'test_inference_sec': test_infer_time
    }
])
print(results_df)

In [ ]:
analysis_df = test_df[['text', 'text_clean', 'label', 'label_name', 'char_len', 'word_len']].copy()
analysis_df['pred'] = test_preds
analysis_df['pred_name'] = analysis_df['pred'].map(id2label)
analysis_df['correct'] = analysis_df['label'] == analysis_df['pred']
analysis_df['confidence'] = test_probas.max(axis=1)
analysis_df['true_label_prob'] = test_probas[np.arange(len(test_probas)), analysis_df['label'].values]
analysis_df['pred_label_prob'] = test_probas[np.arange(len(test_probas)), analysis_df['pred'].values]

sorted_probas = np.sort(test_probas, axis=1)
analysis_df['second_best_prob'] = sorted_probas[:, -2]
analysis_df['margin_top1_top2'] = analysis_df['pred_label_prob'] - analysis_df['second_best_prob']
analysis_df['error_type'] = np.where(
    analysis_df['correct'],
    'correct',
    np.where(analysis_df['confidence'] >= 0.80, 'high_confidence_error', 'lower_confidence_error')
)

print('Total test examples:', len(analysis_df))
print('Correct predictions:', int(analysis_df['correct'].sum()))
print('Misclassified test examples:', int((~analysis_df['correct']).sum()))
print('\nConfidence summary on test set:')
print(analysis_df[['confidence', 'true_label_prob', 'margin_top1_top2']].describe())

In [ ]:
misclassified = analysis_df[~analysis_df['correct']].copy()

confusion_pairs = (
    misclassified.groupby(['label_name', 'pred_name'])
    .size()
    .reset_index(name='count')
    .sort_values(['count', 'label_name', 'pred_name'], ascending=[False, True, True])
)

print('Top confusion pairs:')
print(confusion_pairs.head(25).to_string(index=False))

In [ ]:
per_class_failure = (
    analysis_df.groupby('label_name')
    .agg(
        support=('label_name', 'size'),
        correct=('correct', 'sum'),
        avg_confidence=('confidence', 'mean'),
        avg_true_label_prob=('true_label_prob', 'mean')
    )
    .reset_index()
)
per_class_failure['errors'] = per_class_failure['support'] - per_class_failure['correct']
per_class_failure['accuracy'] = per_class_failure['correct'] / per_class_failure['support']
per_class_failure['error_rate'] = per_class_failure['errors'] / per_class_failure['support']
per_class_failure = per_class_failure.sort_values(['error_rate', 'errors'], ascending=[False, False])

print('Per-class failure summary:')
print(per_class_failure.to_string(index=False))

In [ ]:
hardest_examples = analysis_df.sort_values(['true_label_prob', 'margin_top1_top2', 'confidence'], ascending=[True, True, True]).copy()
print('Hardest examples by lowest probability assigned to the true label:')
print(
    hardest_examples[
        ['text', 'label_name', 'pred_name', 'correct', 'confidence', 'true_label_prob', 'second_best_prob', 'margin_top1_top2', 'word_len']
    ].head(30).to_string(index=False)
)

In [ ]:
high_conf_errors = misclassified.sort_values(['confidence', 'margin_top1_top2'], ascending=[False, False]).copy()
print('Highest-confidence mistakes:')
print(
    high_conf_errors[
        ['text', 'label_name', 'pred_name', 'confidence', 'true_label_prob', 'second_best_prob', 'margin_top1_top2', 'word_len']
    ].head(30).to_string(index=False)
)

In [ ]:
low_margin_cases = analysis_df.sort_values(['margin_top1_top2', 'confidence'], ascending=[True, True]).copy()
print('Most ambiguous examples by smallest top-1 vs top-2 margin:')
print(
    low_margin_cases[
        ['text', 'label_name', 'pred_name', 'correct', 'confidence', 'true_label_prob', 'second_best_prob', 'margin_top1_top2', 'word_len']
    ].head(30).to_string(index=False)
)

In [ ]:
print('Representative misclassifications for each true class:')
for class_name in label_names:
    subset = misclassified[misclassified['label_name'] == class_name].copy()
    subset = subset.sort_values(['confidence', 'true_label_prob'], ascending=[False, True]).head(5)
    print('\nTRUE CLASS:', class_name)
    if len(subset) == 0:
        print('No misclassifications for this class in the test set.')
    else:
        print(subset[['text', 'pred_name', 'confidence', 'true_label_prob', 'margin_top1_top2', 'word_len']].to_string(index=False))

In [ ]:
print('Representative misclassifications for each confusion pair:')
top_pairs = confusion_pairs.head(10)
for _, row in top_pairs.iterrows():
    true_name = row['label_name']
    pred_name = row['pred_name']
    pair_df = misclassified[(misclassified['label_name'] == true_name) & (misclassified['pred_name'] == pred_name)].copy()
    pair_df = pair_df.sort_values(['confidence', 'true_label_prob'], ascending=[False, True]).head(3)
    print(f'\nPAIR: true={true_name} -> pred={pred_name} | count={row["count"]}')
    print(pair_df[['text', 'confidence', 'true_label_prob', 'margin_top1_top2', 'word_len']].to_string(index=False))

In [ ]:
class_confidence_summary = (
    analysis_df.groupby(['label_name', 'correct'])
    .agg(
        n=('text', 'size'),
        mean_confidence=('confidence', 'mean'),
        median_confidence=('confidence', 'median'),
        mean_true_label_prob=('true_label_prob', 'mean'),
        mean_margin=('margin_top1_top2', 'mean')
    )
    .reset_index()
    .sort_values(['label_name', 'correct'])
)
print('Confidence behavior by class and correctness:')
print(class_confidence_summary.to_string(index=False))

In [ ]:
summary_rows = []
for class_name in label_names:
    subset = analysis_df[analysis_df['label_name'] == class_name].copy()
    wrong = subset[~subset['correct']].copy()
    top_wrong_target = None
    top_wrong_target_count = 0
    if len(wrong) > 0:
        pair_counts = wrong['pred_name'].value_counts()
        top_wrong_target = pair_counts.index[0]
        top_wrong_target_count = int(pair_counts.iloc[0])
    summary_rows.append({
        'class_name': class_name,
        'support': len(subset),
        'accuracy': subset['correct'].mean(),
        'mean_confidence': subset['confidence'].mean(),
        'mean_true_label_prob': subset['true_label_prob'].mean(),
        'top_confused_with': top_wrong_target,
        'top_confused_with_count': top_wrong_target_count
    })

class_summary_df = pd.DataFrame(summary_rows).sort_values('accuracy')
print('Compact per-class summary with dominant failure destination:')
print(class_summary_df.to_string(index=False))

In [ ]:
def predict_emotion(texts):
    cleaned = [str(t).strip().lower() for t in texts]
    pred_ids = model.predict(cleaned)
    probas = model.predict_proba(cleaned)
    confs = probas.max(axis=1)
    sorted_probs = np.sort(probas, axis=1)
    return pd.DataFrame({
        'text': texts,
        'pred_label_id': pred_ids,
        'pred_label': [id2label[i] for i in pred_ids],
        'confidence': confs,
        'second_best_prob': sorted_probs[:, -2],
        'margin_top1_top2': confs - sorted_probs[:, -2]
    })

sample_texts = [
    'i feel amazing and grateful today',
    'i am really upset and angry about what happened',
    'i miss my friends and feel lonely',
    'i am scared about tomorrow',
    'this was such a lovely surprise'
]

print(predict_emotion(sample_texts).to_string(index=False))